# B3 · Gruplara ayırıp her grubun ortalamasını almak (groupby)

**Bugünün tek fikri:** tabloyu bir sütundaki değerlere göre **gruplara ayır**, her grubun içinde **bir hesap** yap (ortalama, sayı…).

Gündelik benzetme: sınıftaki sınav kâğıtlarını şubelere göre üç yığına ayırıyorsun, sonra her yığının ortalamasını alıyorsun. Ayır → hesapla → sonuçları yan yana koy.

B2'nin kapanışındaki sorunu bugün cevaplıyoruz: *GFS hangi mevsimde, hangi saatte en çok şaşıyor; hafif rüzgârda tutarlı mı?*

**Kural:** her kod hücresinden önce altındaki kutuya tek cümle yaz, sonra çalıştır.

## 1. Oyuncak tablo

6 satır. `hata` = gfs − ölçüm (B2'deki gibi; eksi ise GFS az tahmin etmiş).

In [ ]:
import pandas as pd

oyuncak = pd.DataFrame({
    "mevsim": ["kis", "yaz", "kis", "yaz", "yaz", "ilkbahar"],
    "hata":   [-1,    -6,    -3,    -4,    -5,    -2],
})
oyuncak

✍️ *Bu hücre ne yapıyor (tek cümle):*

### 1a. Önce elle: düz `for` döngüsü

groupby'ın içinde ne olduğunu görmek için aynı işi önce döngüyle yapıyoruz. Her mevsim için: o mevsimin satırlarını maskeyle seç (B1), `hata` ortalamasını al.

`.unique()` sütundaki farklı değerleri bir kez verir: `kis, yaz, ilkbahar`.

In [ ]:
for m in oyuncak["mevsim"].unique():
    secili = oyuncak[oyuncak["mevsim"] == m]
    print(m, secili["hata"].mean())

✍️ *Bu hücre ne yapıyor (tek cümle):*

🧮 *Çalıştırmadan önce kafadan hesapla: kış kaç çıkmalı, yaz kaç?*

### 1b. Aynı iş tek satırda: `groupby`

Okuması soldan sağa: **mevsime göre grupla** → **`hata` sütununu al** → **ortalamasını hesapla**.

```
tablo.groupby("grup_sütunu")["hesap_sütunu"].mean()
```

In [ ]:
oyuncak.groupby("mevsim")["hata"].mean()

✍️ *Bu hücre ne yapıyor (tek cümle):*

🔍 *1a'daki döngüyle aynı sayılar mı? Sıra neden farklı?*

### 1c. Ortalama yerine başka hesap

`.mean()` yerine `.count()` (kaç satır), `.max()`, `.min()` de yazılır. Kalıp aynı, yalnız sondaki değişiyor.

In [ ]:
oyuncak.groupby("mevsim")["hata"].count()

✍️ *Bu hücre ne yapıyor (tek cümle):*

### 1d. Sonucu sıralamak

groupby'ın sonucu bir Series (B1b). B2'deki `sort_values` ona da uyar; Series'te sütun adı vermene gerek yok.

In [ ]:
oyuncak.groupby("mevsim")["hata"].mean().sort_values()

✍️ *Bu hücre ne yapıyor (tek cümle):*

## 2. Gerçek veri

İlk hücre B2'de senin ürettiğin sütunları yeniden ekliyor (yeni defter, tablo sıfırdan yükleniyor). Bunu Pezo yazdı, sadece çalıştır.

In [ ]:
from ruzgar.data import load_aligned
veri = load_aligned()

mevsim_sozluk = {12: "kis", 1: "kis", 2: "kis",
                 3: "ilkbahar", 4: "ilkbahar", 5: "ilkbahar",
                 6: "yaz", 7: "yaz", 8: "yaz",
                 9: "sonbahar", 10: "sonbahar", 11: "sonbahar"}

veri["mutlak_hata"] = veri["hata"].abs()
veri["yerel_saat"] = (veri["time"].dt.hour + 3) % 24
veri["mevsim"] = veri["time"].dt.month.map(mevsim_sozluk)
veri["sinif"] = pd.cut(veri["speed_kn_obs"], bins=[0, 10, 15, 20, 100],
                       labels=["hafif", "orta", "iyi", "kuvvetli"], right=False)
veri.head()

### Alıştırma 1 · GFS hangi mevsimde en çok şaşıyor?

Mevsime göre grupla, `mutlak_hata` ortalamasını al, küçükten büyüğe sırala.

In [ ]:
# 1b + 1d kalıbı: veri.groupby(...)[...].mean().sort_values()

✍️ *En kötü mevsim hangisi? Sence neden (ipucu: yazın İzmir'de hangi rüzgâr eser)?*

### Alıştırma 2 · Az mı tahmin ediyor, çok mu?

Aynı gruplama, ama bu sefer **işaretli** `hata` ortalaması. Eksi = GFS az tahmin ediyor.

In [ ]:
# Alıştırma 1'deki satırı kopyala, yalnız sütun adını değiştir

✍️ *Dört mevsimin işareti aynı mı? 1'deki mutlak hatayla ne kadar yakın?*

### Alıştırma 3 · Günün hangi saatinde?

`yerel_saat`'e göre grupla, `mutlak_hata` ortalaması. Bu sefer sıralama yerine saat sırasıyla bak (groupby zaten grup adına göre sıralı verir).

In [ ]:
# grup sütunu değişiyor, gerisi aynı

✍️ *GFS gece mi, öğleden sonra mı daha iyi? En kötü saat kaç?*

### Alıştırma 4 · Hafif rüzgârda tutarlı mı?

`sinif`'e göre grupla, `hata` ortalaması. 

⚠️ `sinif` bir *kategori* sütunu (`pd.cut` öyle üretiyor); groupby'a `observed=True` ekle, yoksa uyarı verir: `veri.groupby("sinif", observed=True)[...]`

In [ ]:
# 

✍️ *Rüzgâr kuvvetlendikçe hata nasıl değişiyor? Hafifte GFS'e güvenilir mi?*

### Alıştırma 5 (isteğe bağlı) · Ortalama mı, tutarlılık mı?

"Tutarlı" demek yalnız ortalamanın sıfıra yakın olması değil, hataların da **az dağılması**. `.std()` (standart sapma) dağılmayı ölçer: küçükse hatalar hep birbirine yakın. 4'teki satırın sonunu `.std()` yap.

In [ ]:
# 

✍️ *Hangi sınıfta GFS en tutarlı?*

## 3. Kapanış (2 cümle)

✍️ *groupby'ı kendi cümlemle anlatırsam:*

✍️ *Bugünkü sonuçlar rüzgâr modeline ne söylüyor? (ipucu: model hangi durumlarda GFS'i en çok düzeltmek zorunda?)*